# Session 3 — Supabase & Database
## Empire & Ink: AI Mughal Art Studio

**What you'll build:** A Python module that connects to a real Supabase (PostgreSQL) database, inserts gallery records, queries them back, and deletes them — all the CRUD your app needs.

---

## Setup

In [ ]:
!pip install supabase python-dotenv

## Theory 1 — Tables, Rows & Columns

A **database** is like a spreadsheet workbook:
- A **table** is a sheet (e.g. `galleries`)
- A **row** is one record (one painting generation)
- A **column** is a field (title, image_url, user_id…)

| id | user_id | prompt | image_url | style | created_at |
|----|---------|--------|-----------|-------|------------|
| 1  | user123 | A tiger | https://… | akbari | 2025-01-01 |

In [ ]:
galleries_table = []

def insert(record):
    galleries_table.append(record)
    return record

def select_all():
    return galleries_table

def delete_by_id(item_id):
    global galleries_table
    galleries_table = [r for r in galleries_table if r["id"] != item_id]

insert({"id": 1, "prompt": "A tiger in a garden", "style": "akbari"})
insert({"id": 2, "prompt": "Peacock on a terrace", "style": "jahangiri"})
print("All:", select_all())
delete_by_id(1)
print("After delete:", select_all())

## Theory 2 — SQL Basics

```sql
CREATE TABLE galleries (
    id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    user_id TEXT NOT NULL,
    prompt TEXT NOT NULL,
    image_url TEXT,
    style TEXT DEFAULT 'akbari',
    created_at TIMESTAMPTZ DEFAULT NOW()
);

INSERT INTO galleries (user_id, prompt, style)
VALUES ('user123', 'A tiger in a Mughal garden', 'akbari');

SELECT * FROM galleries WHERE user_id = 'user123';

DELETE FROM galleries WHERE id = 'some-uuid';
```

In [ ]:
create_sql = (
    "CREATE TABLE IF NOT EXISTS galleries (\n"
    "    id UUID PRIMARY KEY DEFAULT gen_random_uuid(),\n"
    "    user_id TEXT NOT NULL,\n"
    "    prompt TEXT NOT NULL,\n"
    "    enhanced_prompt TEXT,\n"
    "    image_url TEXT,\n"
    "    style TEXT DEFAULT 'akbari',\n"
    "    generation_type TEXT DEFAULT 'generate',\n"
    "    created_at TIMESTAMPTZ DEFAULT NOW()\n"
    ");"
)
print(create_sql)

## Theory 3 — Row Level Security (RLS)

RLS ensures each user only sees their own rows:

```sql
ALTER TABLE galleries ENABLE ROW LEVEL SECURITY;

CREATE POLICY "Users see own gallery"
ON galleries FOR SELECT
USING (user_id = auth.uid()::text);
```

Think of RLS as a bouncer checking "is this your row?" before letting you in.

In [ ]:
def query_with_rls(table, current_user_id):
    return [row for row in table if row["user_id"] == current_user_id]

all_rows = [
    {"id": 1, "user_id": "alice", "prompt": "A peacock"},
    {"id": 2, "user_id": "bob",   "prompt": "A tiger"},
    {"id": 3, "user_id": "alice", "prompt": "A falcon"},
]
print("Alice sees:", query_with_rls(all_rows, "alice"))
print("Bob sees:  ", query_with_rls(all_rows, "bob"))

---
## In-Class Exercises

### Exercise 1 — Write the CREATE TABLE SQL

In [ ]:
# YOUR CODE HERE — write CREATE TABLE SQL for the galleries table
create_table_sql = (
    "CREATE TABLE IF NOT EXISTS galleries (\n"
    "    -- YOUR COLUMNS HERE\n"
    ");"
)
print(create_table_sql)

### Exercise 2 — Connect to Supabase

In [ ]:
import os
from supabase import create_client

try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get("SUPABASE_URL")
    SUPABASE_KEY = userdata.get("SUPABASE_KEY")
except Exception:
    SUPABASE_URL = os.getenv("SUPABASE_URL", "")
    SUPABASE_KEY = os.getenv("SUPABASE_KEY", "")

# YOUR CODE HERE — create the Supabase client
if SUPABASE_URL and SUPABASE_KEY:
    supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("Connected to Supabase!")
else:
    supabase = None
    print("Add SUPABASE_URL and SUPABASE_KEY as Colab secrets to continue.")

### Exercise 3 — Insert a test record

In [ ]:
test_record = {
    "user_id": "test-user-001",
    "prompt": "A royal hunt scene with elephants",
    "style": "akbari",
    "image_url": "https://example.com/placeholder.jpg",
}
# YOUR CODE HERE
# result = supabase.table("galleries").insert(test_record).execute()
# print("Inserted:", result.data)
print("Record ready to insert:", test_record)

### Exercise 4 — Query all records for a user

In [ ]:
user_id = "test-user-001"
# YOUR CODE HERE
# result = supabase.table("galleries").select("*").eq("user_id", user_id).execute()
# for row in result.data:
#     print(f"  {row['id']} — {row['prompt']}")
print("Query: SELECT * FROM galleries WHERE user_id =", user_id)

### Exercise 5 — Delete a record by ID

In [ ]:
item_id = "paste-a-real-uuid-here"
# YOUR CODE HERE
# result = supabase.table("galleries").delete().eq("id", item_id).execute()
# print("Deleted:", result.data)
print("Delete: DELETE FROM galleries WHERE id =", item_id)

---
## Post-Class Exercises

### Challenge 1 — save_gallery_item()

In [ ]:
def save_gallery_item(supabase_client, user_id, prompt, enhanced_prompt, image_url, style="akbari"):
    record = {
        "user_id": user_id, "prompt": prompt,
        "enhanced_prompt": enhanced_prompt,
        "image_url": image_url, "style": style,
    }
    # YOUR CODE HERE
    # result = supabase_client.table("galleries").insert(record).execute()
    # return result.data[0] if result.data else None
    return record  # placeholder

print(save_gallery_item(None, "u1", "A peacock", "Resplendent peacock...", "https://ex.com/1.jpg"))

### Challenge 2 — get_user_gallery() with filter

In [ ]:
def get_user_gallery(supabase_client, user_id, limit=20):
    # YOUR CODE HERE — query rows newest first, capped at limit
    # result = supabase_client.table("galleries")\
    #     .select("*").eq("user_id", user_id)\
    #     .order("created_at", desc=True).limit(limit).execute()
    # return result.data
    return []  # placeholder

rows = get_user_gallery(None, "u1")
print(f"Gallery rows returned: {len(rows)}")

### Challenge 3 — delete_gallery_item() returning bool

In [ ]:
def delete_gallery_item(supabase_client, item_id, user_id):
    # YOUR CODE HERE — verify ownership before deleting
    # Return True if deleted, False if not found / wrong user
    pass

result = delete_gallery_item(None, "fake-id", "u1")
print("Deleted:", result)

### Challenge 4 — Full insert / query / delete test

In [ ]:
def test_full_crud(supabase_client):
    print("Step 1: Insert")
    # item = save_gallery_item(...)
    print("  Inserted (placeholder)")
    print("Step 2: Query back")
    # rows = get_user_gallery(...)
    print("  Query OK (placeholder)")
    print("Step 3: Delete")
    # deleted = delete_gallery_item(...)
    print("  Deleted (placeholder)")
    print("Step 4: Confirm gone")
    print("  Confirmed (placeholder)")

test_full_crud(None)

---
## What you built today

- Understood databases as structured tables of rows and columns
- Wrote SQL CREATE, INSERT, SELECT, and DELETE statements
- Connected to Supabase using the Python SDK
- Implemented all four CRUD operations with Row Level Security

**Next session:** Session 4 — Gemini AI & Prompt Engineering — generate your first Mughal art image!